In [56]:
import mysql.connector
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\svi02\\.spyder-py3\\erfpPROD.ini')

['C:\\Users\\svi02\\.spyder-py3\\erfpPROD.ini']

In [57]:
host=config['erfpPROD']['host']
user=config['erfpPROD']['user']
pwd=config['erfpPROD']['pwd']
database=config['erfpPROD']['database']
print("SUCCESS")

SUCCESS


In [58]:
conn = mysql.connector.connect(
          host=host,
          user=user,
          passwd=pwd,
          database=database)
cursor = conn.cursor()

In [90]:
sql_select_query = """SELECT 
      htl.HOTEL_ID_VALUE AS HOTEL_ID
    , htl.HOTEL_NAME_NAME AS HOTEL_NAME
    , rfp.ID_VALUE AS RFP_ID
    , rfp.NAME AS RFP_NAME
    , rfp.SEASON AS RFP_SEASON
    , tr.CURRENCY AS TARGET_RATE_CURRENCY
    , tr.RATE_TYPE
    , tr.BREAKFAST_TYPE
    , tr.AMOUNT AS TARGET_RATE
    , htl.MARKET_PLACE_HOTEL_STATUS
    , htl.AVERAGE_RATE_IN_EUR_AMOUNT
    , rfp.IS_TEST_RFP
    , rfp.STATUS
    ,trim(DATE(rfp.ENTITY_STATUS_CREATING_DATE)) AS RFP_CREATION_DATE
  FROM erfp_target_rate tr
LEFT JOIN erfp_hotel_for_rfpentity htl ON tr.OR_HOTEL_VALUE = htl.ID_VALUE
JOIN erfp_rfpentity rfp ON rfp.ID_VALUE = htl.OR_RFP_VALUE
WHERE rfp.SEASON IN (2019)
    AND htl.MARKET_PLACE_HOTEL_STATUS IN('INVITED', 'SIGNED_UP', 'SOLICITED' ) 
    AND IS_TEST_RFP IN (0)
ORDER BY HOTEL_ID"""

cursor.execute(sql_select_query)

QUERY = cursor.fetchall()
print('Total Row(s):', cursor.rowcount)

# Importing data into a DataFrame
import pandas as pd
erfp_df = pd.DataFrame()
a=[]
for row in QUERY:
        a.append(row)

erfp_df = pd.DataFrame(a)
df_col_names =  [i[0] for i in cursor.description]
erfp_df.columns = df_col_names

Total Row(s): 14261


In [60]:
cursor.description[0][0]

'HOTEL_ID'

In [74]:
erfp_df.head()

,HOTEL_ID,HOTEL_NAME,RFP_ID,RFP_NAME,RFP_SEASON,TARGET_RATE_CURRENCY,RATE_TYPE,BREAKFAST_TYPE,TARGET_RATE,MARKET_PLACE_HOTEL_STATUS,AVERAGE_RATE_IN_EUR_AMOUNT,IS_TEST_RFP,STATUS,RFP_CREATION_DATE
0,9,Ringhotel Birke,0719904f-c4b5-4676-8b91-16dd70014763,Rheinmetall RFP 2019,2019,EUR,NLRA,INCLUDED,84.00,SIGNED_UP,100.05,0,InProgress,2018-09-17
1,9,Ringhotel Birke,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,75.00,SIGNED_UP,75.10,0,InProgress,2018-08-15
2,10,relexa Bellevue,df0351f8-10bd-11e8-940c-005056a93fbf,HRS Corporate Club Rate RFP 2019/2020 DACH,2019,EUR,LRA,EXCLUDED,0.00,SIGNED_UP,None,0,InProgress,2018-02-13
3,10,relexa Bellevue,96cfd8d3-ed77-4cc3-b263-3a4042be167e,Ströer RFP 2019,2019,EUR,LRA,INCLUDED,93.00,SIGNED_UP,104.31,0,InProgress,2018-09-20
4,12,Select Hotel Hamburg Nord,b93660b8-71f1-4b14-83e6-c7920a586012,Danaher EMEA RFP 2019,2019,EUR,LRA,INCLUDED,75.00,SIGNED_UP,82.30,0,InProgress,2018-07-25


In [40]:
syb_df = pd.read_excel('C:/Users/svi02/Documents/misc/target_rates_syb_prod.xlsx',
                       sheet_name='Grid Results', 
                       encoding='utf-8')

In [73]:
import pyexasol
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\svi02\\.spyder-py3\\ExasolPROD.ini')

dsn=config['exasolPROD']['dsn']
user=config['exasolPROD']['user']
pwd=config['exasolPROD']['pwd']
schema=config['exasolPROD']['schema']
print("SUCCESS")

# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
sql_query =  "select * from dwhbil.LKP_CURRENCY_EXCHANGE_RATE_HIST WHERE YEAR(CURRENCY_EXCHANGE_RATE_DATE) >= 2018"
QUERY = connect.execute(sql_query)

# Importing data into a DataFrame
import pandas as pd
exh_df = pd.DataFrame()
a=[]
for row in QUERY:
    a.append(row)
#print(len(a))
exh_df = pd.DataFrame(a)
df_col_names = QUERY.col_names
exh_df.columns = df_col_names
exh_df.head()

SUCCESS


,CURRENCY_ISO,CURRENCY_EXCHANGE_RATE_DATE,CURRENCY_EXCHANGE_RATE
0,MTL,2018-07-25,0.4293
1,MXP,2018-07-25,21.98019
2,NGN,2018-07-25,423.03341
3,NZD,2018-07-25,1.71736
4,PHP,2018-07-25,62.39458


In [94]:
add_exch = pd.merge(left = erfp_df, 
                    right = exh_df, 
                    left_on = ['RFP_CREATION_DATE','TARGET_RATE_CURRENCY'], 
                    right_on = ['CURRENCY_EXCHANGE_RATE_DATE','CURRENCY_ISO'], 
                    how = 'inner')
add_exch.TARGET_RATE = add_exch.TARGET_RATE.astype
add_exch.CURRENCY_EXCHANGE_RATE = float(add_exch.CURRENCY_EXCHANGE_RATE)

TypeError: cannot convert the series to <class 'float'>

In [96]:
add_exch['TARGET_RATE_EUR'] = add_exch.TARGET_RATE.astype('float64')/add_exch.CURRENCY_EXCHANGE_RATE.astype('float64')
add_exch.head(10)

,HOTEL_ID,HOTEL_NAME,RFP_ID,RFP_NAME,RFP_SEASON,TARGET_RATE_CURRENCY,RATE_TYPE,BREAKFAST_TYPE,TARGET_RATE,MARKET_PLACE_HOTEL_STATUS,AVERAGE_RATE_IN_EUR_AMOUNT,IS_TEST_RFP,STATUS,RFP_CREATION_DATE,CURRENCY_ISO,CURRENCY_EXCHANGE_RATE_DATE,CURRENCY_EXCHANGE_RATE,TARGET_RATE_EUR
0,9,Ringhotel Birke,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,75.00,SIGNED_UP,75.10,0,InProgress,2018-08-15,EUR,2018-08-15,1,75.0
1,32,Brenner,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,69.00,SIGNED_UP,57.30,0,InProgress,2018-08-15,EUR,2018-08-15,1,69.0
2,35,City-Hotel,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,73.00,SIGNED_UP,73.48,0,InProgress,2018-08-15,EUR,2018-08-15,1,73.0
3,40,Jung,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,55.00,SIGNED_UP,62.26,0,InProgress,2018-08-15,EUR,2018-08-15,1,55.0
4,42,H+ Hotel Bochum,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,70.00,SIGNED_UP,69.48,0,InProgress,2018-08-15,EUR,2018-08-15,1,70.0
5,56,Astoria,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,72.00,SIGNED_UP,75.25,0,InProgress,2018-08-15,EUR,2018-08-15,1,72.0
6,67,Diehls Hotel,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,75.00,SIGNED_UP,75.14,0,InProgress,2018-08-15,EUR,2018-08-15,1,75.0
7,97,Advena Europa,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,73.00,SIGNED_UP,75.11,0,InProgress,2018-08-15,EUR,2018-08-15,1,73.0
8,110,Mercure Hotel Saarbruecken Sued,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,74.00,SIGNED_UP,74.09,0,InProgress,2018-08-15,EUR,2018-08-15,1,74.0
9,136,Schlosshotel,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,75.00,SIGNED_UP,75.19,0,InProgress,2018-08-15,EUR,2018-08-15,1,75.0


In [97]:
result_df = pd.merge(left = add_exch, right = syb_df, on = ['RFP_ID', 'HOTEL_ID'], how = 'left')
result_df.head(100)

,HOTEL_ID,HOTEL_NAME_x,RFP_ID,RFP_NAME_x,RFP_SEASON,TARGET_RATE_CURRENCY,RATE_TYPE,BREAKFAST_TYPE,TARGET_RATE,MARKET_PLACE_HOTEL_STATUS,...,HOTEL_CITY_NAME,HOTEL_COUNTRY_NAME,RFP_ID.1,HOTEL_ID.1,ACCEPTED_RATE_TYPE_CD,AMOUNT_OFFER_LRA_INCL_EUR,AMOUNT_OFFER_LRA_EXCL_EUR,AMOUNT_OFFER_NLRA_INCL_EUR,AMOUNT_OFFER_NLRA_EXCL_EUR,RFP_OFFER_TIMESTAMP
0,9,Ringhotel Birke,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,75.00,SIGNED_UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
1,32,Brenner,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,69.00,SIGNED_UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2,35,City-Hotel,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,73.00,SIGNED_UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
3,40,Jung,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,55.00,SIGNED_UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
4,42,H+ Hotel Bochum,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,70.00,SIGNED_UP,...,Bochum (North Rhine-Westphalia),Germany,a110b8f7-6aa9-446f-9efa-953c6ca637d6,42.0,LRA_INCL,70.0,NaN,NaN,NaN,2019-01-08 14:34:08
5,56,Astoria,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,72.00,SIGNED_UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
6,67,Diehls Hotel,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,75.00,SIGNED_UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
7,97,Advena Europa,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,73.00,SIGNED_UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
8,110,Mercure Hotel Saarbruecken Sued,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,74.00,SIGNED_UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
9,136,Schlosshotel,a110b8f7-6aa9-446f-9efa-953c6ca637d6,Deutsche Telekom Group RFP 2019 Germany,2019,EUR,LRA,INCLUDED,75.00,SIGNED_UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


In [98]:
# Write in excel file
file = "C:\\Users\\svi02\\Documents\\misc\\target_rates_normal.xlsx" 
result_df.to_excel(file, sheet_name='Target_rates', header=True, encoding='utf-8', index=False, freeze_panes=(1,0))
    